<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M01/M01_Lab2_Parameters_Tokens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🎛️ M01 Lab 2 — Parameters, Tokens & Cost</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐ Difficulty: Beginner &nbsp;|&nbsp; ⏱ Time: ~12 min</p>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 35px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 28px;">M01 Lab 2 — Model Parameters & Token Economics</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Northeastern University</p>
  <p style="color: rgba(255,255,255,0.7); margin: 4px 0 0; font-size: 13px;">Difficulty: ⭐ &nbsp;|&nbsp; Time: ~12 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Understand how <code>temperature</code> controls output randomness</li>
    <li>Use <code>max_tokens</code> to control response length</li>
    <li>Understand <b>tokenization</b> — how LLMs break text into pieces</li>
    <li>Estimate <b>API costs</b> for different models and use cases</li>
  </ol>
</div>

In [ ]:
# ============================================================
# 📦 Install & Setup
# ============================================================
!pip install -q --upgrade openai tiktoken
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
from openai import OpenAI
from dads5250 import setup_openai, show_response, compare_responses, count_tokens, estimate_cost, quiz
import tiktoken

client = setup_openai()

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ Temperature: Controlling Randomness</h2>
</div>

The `temperature` parameter is one of the most important controls you have over LLM output.

| Temperature | Behavior | Best For |
|:-----------:|----------|----------|
| **0.0** | Deterministic — same input → same output every time | Factual Q&A, data extraction, classification |
| **0.3–0.7** | Balanced — some variation while staying coherent | General-purpose tasks, summarization |
| **0.8–1.5** | Creative — more diverse, surprising outputs | Brainstorming, creative writing, ideation |
| **1.5–2.0** | Very random — can become incoherent | Rarely useful in production |

Let's see this in action:

In [ ]:
# ============================================================
# 🌡️ Temperature Comparison
# ============================================================

prompt = "Suggest a creative name for an AI-powered coffee shop."

print(f"📝 Prompt: \"{prompt}\"\n")

for temp in [0.0, 0.5, 1.0, 1.5]:
    # Run the same prompt 2 times at each temperature to show consistency vs variation
    results = []
    for _ in range(2):
        r = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=30,
        )
        results.append(r.choices[0].message.content.strip())

    same = "✅ Same" if results[0] == results[1] else "🔀 Different"
    print(f"🌡️ temp={temp:.1f} → {same}")
    print(f"   Run 1: {results[0]}")
    print(f"   Run 2: {results[1]}")
    print()

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> At <code>temp=0.0</code>, both runs should produce identical output. As temperature increases, you'll see more variation. In production, use <b>low temperature</b> for consistency (e.g., customer support) and <b>higher temperature</b> for creativity (e.g., marketing copy).
</div>

### `max_tokens`: Controlling Response Length

The `max_tokens` parameter caps how long the response can be. The model will stop generating once it hits this limit — even mid-sentence.

In [ ]:
# ============================================================
# 📏 max_tokens Comparison
# ============================================================

prompt = "Explain the concept of supply and demand in economics."

for max_tok in [20, 50, 150]:
    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tok,
    )
    text = r.choices[0].message.content.strip()
    print(f"📏 max_tokens={max_tok:>3} → ({r.usage.completion_tokens} tokens used)")
    print(f"   {text}")
    print()

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Tokenization: How LLMs See Text</h2>
</div>

LLMs don't process raw text — they break it into **tokens** (subword units). Understanding tokenization helps you:

- **Estimate costs** before running expensive prompts
- **Stay within context limits** (e.g., gpt-4.1-mini has a 128K token window)
- **Optimize prompt length** for efficiency

In [ ]:
# ============================================================
# 🧩 Visualizing Tokenization
# ============================================================

enc = tiktoken.encoding_for_model("gpt-4.1-mini")

examples = [
    "Hello world",
    "Generative AI is transforming how we build software.",
    "supercalifragilisticexpialidocious",
    "The price is $129.99 (tax included).",
    "こんにちは世界",  # "Hello world" in Japanese
]

for text in examples:
    tokens = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"📝 \"{text}\"")
    print(f"   🧩 {len(tokens)} tokens → {decoded}")
    print()

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Insights:</b>
  <ul style="margin: 6px 0 0;">
    <li>Common English words = 1 token. Uncommon/long words get split into multiple tokens.</li>
    <li>Numbers, punctuation, and special characters have their own tokens.</li>
    <li>Non-English text (Japanese, Arabic, etc.) uses <b>more tokens per character</b> — important for multilingual apps.</li>
    <li>Rule of thumb: <b>1 token ≈ 4 characters</b> in English, or about <b>¾ of a word</b>.</li>
  </ul>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ API Cost Estimation</h2>
</div>

LLM APIs charge per token. Different models have vastly different pricing. Let's compare:

In [ ]:
# ============================================================
# 💰 Cost Estimation Across Models
# ============================================================
from dads5250.cost import PRICING

print("💰 Current Pricing (per 1M tokens):\n")
print(f"{'Model':<22} {'Input':>8} {'Output':>8}")
print("-" * 40)
for model, (in_rate, out_rate) in sorted(PRICING.items()):
    print(f"{model:<22} ${in_rate:>6.2f}  ${out_rate:>6.2f}")

# Real scenario: estimate cost for a typical chatbot
print("\n" + "=" * 40)
print("📊 Scenario: Customer support chatbot")
print("   1,000 conversations/day")
print("   ~500 input + 200 output tokens each")
print("=" * 40)

for model in ["gpt-4.1-nano", "gpt-4.1-mini", "gpt-4.1", "gpt-4o"]:
    if model in PRICING:
        cost = estimate_cost(500 * 1000, 200 * 1000, model)
        daily = cost["total_cost"]
        print(f"\n   {model}:")
        print(f"     Daily:   ${daily:.2f}")
        print(f"     Monthly: ${daily * 30:.2f}")

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> <code>gpt-4.1-mini</code> is the sweet spot for most applications — it's 5x cheaper than <code>gpt-4.1</code> while being nearly as capable for standard tasks. Use the full <code>gpt-4.1</code> only when you need maximum reasoning quality.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
quiz([
    {
        "q": "What does the temperature parameter control?",
        "options": [
            "The speed of the API response",
            "The maximum length of the output",
            "The randomness and creativity of the output",
            "The cost per token"
        ],
        "answer": 2,
        "explanation": "Temperature controls randomness: 0 = deterministic (same output each time), higher values = more varied and creative outputs."
    },
    {
        "q": "Approximately how many tokens is the sentence 'Hello, how are you doing today?' in English?",
        "options": [
            "3 tokens",
            "7-8 tokens",
            "15 tokens",
            "30 tokens"
        ],
        "answer": 1,
        "explanation": "In English, ~1 token per word is a rough estimate. This 7-word sentence is approximately 7-8 tokens (punctuation adds a token too)."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise: Temperature Experiment</h2>
</div>

Run the **same prompt** at 3 different temperatures and document how the outputs change.

In [ ]:
# ============================================================
# 🔧 Exercise: Temperature Experiment
# ============================================================

prompt = "Write a one-sentence tagline for a university course on Generative AI."
results = {}

for temp in [0.0, 0.5, 1.0]:
    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=60,
    )
    results[f"🌡️ temp={temp}"] = r.choices[0].message.content.strip()

compare_responses(results)

**📝 Your Observations:** *(double-click to edit)*

| Temperature | Output Style | Creativity Level |
|:-----------:|-------------|------------------|
| 0.0 | _[Your observation]_ | _[Low / Medium / High]_ |
| 0.5 | _[Your observation]_ | _[Low / Medium / High]_ |
| 1.0 | _[Your observation]_ | _[Low / Medium / High]_ |

**Overall pattern:** _[Describe what you noticed about how temperature affects output]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these experiments on your own:</p>
  <ol style="font-size: 14px;">
    <li>Count tokens for a paragraph in your native language vs English — how do they compare?</li>
    <li>Estimate the monthly API cost if your startup makes 10,000 API calls/day with ~300 tokens each</li>
    <li>Try temperature 2.0 — what happens to the output quality?</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 2 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You now understand the key levers for controlling LLM behavior and cost.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><b>Temperature</b>: 0 = deterministic, 1+ = creative. Choose based on your use case.</li>
      <li><b>max_tokens</b>: Caps response length. Set it to avoid runaway costs.</li>
      <li><b>Tokens ≈ ¾ words</b> in English. Non-English text uses more tokens per character.</li>
      <li><b>gpt-4.1-mini</b> is the cost-effective default — pennies per 1000 calls.</li>
    </ul>
  </div>
</div>

---

## 📋 Assignment M01

Run **3 different prompts** of your choice, each at temperatures **0, 0.5, and 1.0** (9 total API calls).

**Submit:**
1. A table showing all 9 outputs (prompt × temperature)
2. A 1-paragraph observation on how temperature affects the responses
3. Token count and estimated cost for all 9 calls combined

Use the code patterns from this lab as your starting point.